# Argus AI — Phase 2: Flood Classification (ResNet)
Dataset: dhawalsrivastava2583/flood-classification-dataset (Kaggle)

Run all cells top to bottom. Use GPU runtime: Runtime > Change runtime type > T4 GPU

This notebook saves checkpoints to Google Drive from the start, so a disconnect never costs you more than a few epochs.

In [ ]:
# 1. Install dependencies
!pip install kagglehub torch torchvision -q

In [ ]:
# 2. Mount Google Drive FIRST (before anything else) for checkpointing
from google.colab import drive
drive.mount('/content/drive')

import os
checkpoint_dir = '/content/drive/MyDrive/argus_ai_checkpoints/flood_classifier'
os.makedirs(checkpoint_dir, exist_ok=True)
print("Checkpoint dir ready:", checkpoint_dir)

In [ ]:
# 3. Kaggle API setup
# Go to kaggle.com -> Settings -> API Tokens -> Create Legacy API Key -> downloads kaggle.json
from google.colab import files
uploaded = files.upload()  # select kaggle.json when prompted

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
# 4. Download dataset
import kagglehub
path = kagglehub.dataset_download("dhawalsrivastava2583/flood-classification-dataset")
print("Path to dataset files:", path)

In [ ]:
# 5. Inspect dataset structure
!find {path} -maxdepth 2 -type d
print("---")
import os
for d in os.listdir(path):
    full = os.path.join(path, d)
    if os.path.isdir(full):
        print(d, '->', len(os.listdir(full)), 'files')

In [ ]:
# 6. Build train/val split manually (dataset only has 2 folders: Flood Images, Non Flood Images — no pre-split)
# We split 85% train / 15% val, copying file PATHS only (not files) into a simple list-based dataset
import random
from pathlib import Path

random.seed(42)

flood_dir = None
nonflood_dir = None
for d in os.listdir(path):
    if 'non' in d.lower():
        nonflood_dir = os.path.join(path, d)
    elif 'flood' in d.lower():
        flood_dir = os.path.join(path, d)

print("Flood dir:", flood_dir)
print("Non-flood dir:", nonflood_dir)

flood_files = [str(p) for p in Path(flood_dir).glob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
nonflood_files = [str(p) for p in Path(nonflood_dir).glob('*') if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]

random.shuffle(flood_files)
random.shuffle(nonflood_files)

split_flood = int(0.85 * len(flood_files))
split_nonflood = int(0.85 * len(nonflood_files))

train_files = [(f, 1) for f in flood_files[:split_flood]] + [(f, 0) for f in nonflood_files[:split_nonflood]]
val_files = [(f, 1) for f in flood_files[split_flood:]] + [(f, 0) for f in nonflood_files[split_nonflood:]]

random.shuffle(train_files)
random.shuffle(val_files)

print(f"Train: {len(train_files)} images | Val: {len(val_files)} images")
print(f"Train flood: {split_flood}, Train non-flood: {split_nonflood}")

In [ ]:
# 7. Dataset class + dataloaders
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # avoid crashing on slightly corrupt images

class FloodDataset(Dataset):
    def __init__(self, files, transform=None):
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path, label = self.files[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            # fallback: return a black image if a file is unreadable
            img = Image.new('RGB', (224, 224))
        if self.transform:
            img = self.transform(img)
        return img, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = FloodDataset(train_files, transform=train_transform)
val_dataset = FloodDataset(val_files, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print("Dataloaders ready.")

In [ ]:
# 8. Build ResNet18 model (transfer learning) with class weighting for imbalance
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)  # 2 classes: flood, non-flood
model = model.to(device)

# Class weights to handle imbalance (9296 flood vs 3748 non-flood)
num_flood = split_flood
num_nonflood = split_nonflood
total = num_flood + num_nonflood
weight_flood = total / (2 * num_flood)
weight_nonflood = total / (2 * num_nonflood)
class_weights = torch.tensor([weight_nonflood, weight_flood]).to(device)
print("Class weights [non-flood, flood]:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

In [ ]:
# 9. Training loop WITH Drive checkpointing every epoch (never lose more than 1 epoch)
import time

num_epochs = 20
best_val_acc = 0.0
start_epoch = 0

checkpoint_path = os.path.join(checkpoint_dir, 'last_checkpoint.pt')
best_model_path = os.path.join(checkpoint_dir, 'best_model.pt')

# Resume automatically if a checkpoint already exists
if os.path.exists(checkpoint_path):
    print("Found existing checkpoint, resuming...")
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_acc = ckpt['best_val_acc']
    print(f"Resuming from epoch {start_epoch}, best_val_acc so far: {best_val_acc:.4f}")

for epoch in range(start_epoch, num_epochs):
    epoch_start = time.time()

    # --- Train ---
    model.train()
    running_loss, correct, total_train = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total_train += labels.size(0)

    train_loss = running_loss / total_train
    train_acc = correct / total_train

    # --- Validate ---
    model.eval()
    val_loss, val_correct, total_val = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            total_val += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss = val_loss / total_val
    val_acc = val_correct / total_val
    scheduler.step()

    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | Time: {epoch_time:.1f}s")

    # Save checkpoint EVERY epoch to Drive (small file, cheap to save)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_val_acc': max(best_val_acc, val_acc),
    }, checkpoint_path)

    # Save best model separately
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f"  -> New best model saved (val_acc: {val_acc:.4f})")

print("\nTraining complete. Best val accuracy:", best_val_acc)

In [ ]:
# 10. Final evaluation — confusion matrix, precision, recall, F1
from sklearn.metrics import classification_report, confusion_matrix

model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=['Non-Flood', 'Flood']))
print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

In [ ]:
# 11. Quick inference test on one validation image
import matplotlib.pyplot as plt

test_path, test_label = val_files[0]
img = Image.open(test_path).convert('RGB')
img_tensor = val_transform(img).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model(img_tensor)
    probs = torch.softmax(output, dim=1)
    pred = torch.argmax(probs, dim=1).item()

labels_map = {0: 'Non-Flood', 1: 'Flood'}
plt.imshow(img)
plt.title(f"True: {labels_map[test_label]} | Predicted: {labels_map[pred]} ({probs[0][pred]*100:.1f}%)")
plt.axis('off')
plt.show()

In [ ]:
# 12. Download the best model to your laptop
from google.colab import files
files.download(best_model_path)

## Next steps
1. Download `best_model.pt` from cell 12
2. Move it into your local repo: `argus-ai/models/flood_classification/best_model.pt`
3. Note down final val accuracy, precision, recall, F1 from cell 10 for your resume/README
4. Move to Phase 3: LangGraph multi-agent layer

## If disconnected mid-training
Just reconnect, remount Drive, reinstall packages, rerun cells 1-9 in order.
Cell 9 automatically detects the existing checkpoint and resumes from the last completed epoch — no manual resume command needed, no risk of the class-mismatch bug from Phase 1.